# HuggingFace 모델 전체 이어받기

사내 프록시 등이 필요한 네트워크에서 PaddleOCR 모델을 통째로 받는 스크립트 (주피터 노트북판).
- **이어받기**: 중단했다가 다시 실행하면 이어서 받음 (HTTP Range)
- **진행률**: 파일별 진행바 + 전체 진행률 + 속도/남은시간 표시

**프록시 지정 방법** (환경변수 — 노트북 실행 전 설정):
```bat
set SNAP2READ_PROXY=http://<프록시주소>:<포트>
jupyter notebook
```
미지정이면 직접 연결(또는 시스템 프록시)로 시도합니다.

**대상 저장소** (snap2read에 필요한 4개 — `REPO`를 바꿔가며 각각 실행):
- `PaddlePaddle/PP-OCRv6_medium_det`
- `PaddlePaddle/PP-OCRv6_medium_rec`
- `PaddlePaddle/PP-LCNet_x1_0_textline_ori`
- `PaddlePaddle/PP-LCNet_x1_0_doc_ori`


In [ ]:
import os
import time
import json
import urllib.request

# ============ 설정 (REPO를 바꿔가며 모델별 실행) ============
REPO = "PaddlePaddle/PP-LCNet_x1_0_doc_ori"
SAVE_DIR = f"./{REPO.split('/')[-1]}"   # 저장 폴더는 저장소 이름과 동일하게 자동 지정
BASE_URL = f"https://huggingface.co/{REPO}/resolve/main"
# ===========================================================

# 프록시 (환경변수 SNAP2READ_PROXY 로만 지정 — 내부 주소를 저장소에 기록하지 않음)
PROXY = os.environ.get("SNAP2READ_PROXY", "")
if PROXY:
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({'http': PROXY, 'https': PROXY}))
    print(f"프록시 사용: {PROXY}")
else:
    opener = urllib.request.build_opener()
    print("프록시 미사용 (직접 연결)")

os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
# HuggingFace API로 파일 목록 가져오기
print("📂 파일 목록 조회 중...")
api_url = f"https://huggingface.co/api/models/{REPO}/tree/main"
req = urllib.request.Request(api_url)
req.add_header('User-Agent', 'Mozilla/5.0')
resp = opener.open(req, timeout=15)
data = json.loads(resp.read().decode('utf-8'))
resp.close()

all_files = sorted(
    [f for f in data if f.get('type') == 'file'],
    key=lambda x: x.get('size', 0)
)

total_all_size = sum(f['size'] for f in all_files)
print(f"총 {len(all_files)}개 파일, {total_all_size/1024**3:.2f} GB\n")

# 이미 완료된 파일 크기 계산
completed_size = 0
for f in all_files:
    save_path = os.path.join(SAVE_DIR, f['path'])
    if os.path.exists(save_path) and os.path.getsize(save_path) == f['size']:
        completed_size += f['size']

print(f"이미 완료: {completed_size/1024**3:.2f} GB\n")


In [ ]:
def format_time(seconds):
    if seconds < 60:
        return f"{seconds:.0f}초"
    elif seconds < 3600:
        return f"{seconds/60:.1f}분"
    else:
        return f"{seconds/3600:.1f}시간"


def download_file(file_info, global_downloaded, global_total):
    """단일 파일 이어받기"""
    filename = file_info['path']
    file_size = file_info['size']
    save_path = os.path.join(SAVE_DIR, filename)
    url = f"{BASE_URL}/{filename}"

    # 이미 완료된 경우 스킵
    if os.path.exists(save_path) and os.path.getsize(save_path) == file_size:
        print(f"  ✅ 이미 완료: {filename}")
        return True

    # 부분 다운로드 크기
    downloaded = os.path.getsize(save_path) if os.path.exists(save_path) else 0

    # 이어받기 요청
    req = urllib.request.Request(url)
    req.add_header('User-Agent', 'Mozilla/5.0')
    if downloaded > 0:
        req.add_header('Range', f'bytes={downloaded}-')

    try:
        resp = opener.open(req, timeout=60)
    except Exception as e:
        print(f"  ❌ 연결 실패: {e}")
        return False

    # Range 응답 확인
    if downloaded > 0 and resp.status != 206:
        print(f"  ⚠️ 이어받기 불가, 처음부터 다시 받습니다.")
        downloaded = 0

    mode = 'ab' if downloaded > 0 and resp.status == 206 else 'wb'
    chunk_size = 1024 * 1024  # 1MB
    start_time = time.time()
    last_print_time = start_time
    last_downloaded = downloaded

    if downloaded > 0:
        print(f"  📥 이어받기: {downloaded/1024**3:.2f} GB 부터")
    else:
        print(f"  📥 새로 받기: {filename}")

    try:
        with open(save_path, mode) as f:
            while True:
                chunk = resp.read(chunk_size)
                if not chunk:
                    break
                f.write(chunk)
                downloaded += len(chunk)
                global_downloaded += len(chunk)

                # 진행률 표시 (0.5초마다)
                now = time.time()
                if now - last_print_time >= 0.5:
                    elapsed = now - start_time
                    speed = (downloaded - last_downloaded) / (now - last_print_time) if now > last_print_time else 0
                    file_progress = downloaded / file_size * 100
                    global_progress = global_downloaded / global_total * 100
                    remaining_file = (file_size - downloaded) / speed if speed > 0 else 0

                    bar_len = 30
                    filled = int(bar_len * file_progress / 100)
                    bar = '█' * filled + '░' * (bar_len - filled)

                    print(
                        f"\r  [{bar}] {file_progress:.1f}% | "
                        f"{downloaded/1024**3:.2f}/{file_size/1024**3:.2f} GB | "
                        f"{speed/1024**2:.1f} MB/s | "
                        f"전체: {global_progress:.1f}% | "
                        f"남은: {format_time(remaining_file)}",
                        end='', flush=True
                    )
                    last_print_time = now
                    last_downloaded = downloaded

    except KeyboardInterrupt:
        resp.close()
        print(f"\n  ⏸️ 중단됨. {downloaded/1024**3:.2f} GB 까지 받음.")
        return False
    except Exception as e:
        resp.close()
        print(f"\n  ❌ 오류: {e}")
        return False

    resp.close()
    print(f"\n  ✅ 완료: {filename} ({downloaded/1024**3:.2f} GB)")
    return True


In [ ]:
# 전체 다운로드 실행
print("=" * 70)
print(f"🚀 전체 다운로드 시작: {len(all_files)}개 파일, {total_all_size/1024**3:.2f} GB")
print("=" * 70)

global_downloaded = completed_size
global_total = total_all_size
overall_start = time.time()

for i, f in enumerate(all_files, 1):
    print(f"\n[{i}/{len(all_files)}] {f['path']} ({f['size']/1024**3:.2f} GB)")
    success = download_file(f, global_downloaded, global_total)

    # 완료된 파일은 전체 크기에 반영
    save_path = os.path.join(SAVE_DIR, f['path'])
    if os.path.exists(save_path):
        global_downloaded = max(global_downloaded, sum(
            os.path.getsize(os.path.join(SAVE_DIR, ff['path']))
            for ff in all_files[:i]
            if os.path.exists(os.path.join(SAVE_DIR, ff['path']))
        ))

# 최종 결과
elapsed = time.time() - overall_start
print("\n" + "=" * 70)
print(f"🎉 다운로드 완료!")
print(f"   위치: {os.path.abspath(SAVE_DIR)}")
print(f"   소요 시간: {format_time(elapsed)}")
print("=" * 70)
